### 연관규칙(Association Rules) + 네트워크 분석 (국내 KR / 글로벌 GB)

## 개요

Apriori 알고리즘으로 **리뷰 안에서 함께 언급되는 키워드 조합**을 찾고, 이를 네트워크 그래프로 시각화하는 분석입니다.

### 처리 순서
1. 확신도 0.8 이상 리뷰 필터링
2. 정규화 사전 + 불용어 적용 후, 리뷰 한 건을 "키워드 집합(트랜잭션)"으로 변환
3. Apriori로 자주 함께 등장하는 키워드 조합(itemset) 추출 → 연관규칙(A→B) 생성
4. Support/Confidence/Lift 기준으로 필터링 및 정렬
5. 상위 규칙을 네트워크 그래프(PyVis)로 시각화

### 평가지표
- **Support**: 전체 트랜잭션 중 A와 B가 함께 등장하는 비율
- **Confidence**: A 등장 시 B도 함께 등장할 조건부 확률
- **Lift**: 독립 대비 실제 연관 강도 (1 초과 시 양의 연관, 높을수록 강한 패턴)

### ⚠️ 참고 및 실행 전 준비사항
- 정규화 사전(`norm_groups`, `en_norm_groups`)을 만드는 탐색 과정은 생략하고, 최종 사전과 분석 로직만 정리했습니다.
- 불용어 사전 파일(`stopwords.txt`, `stopwords_gb.txt`, `stopwords_gb_sentiment.txt`)이 별도로 필요합니다.
- 표(연관규칙 결과)는 엄격한 기준, 네트워크 시각화는 완화된 기준을 사용해 별도로 재실행했습니다.
- 네트워크 시각화 결과(html)는 리포지토리에 포함하지 않으며, 코드는 참고용으로만 공개합니다.


## 1. 국내(KR) 연관규칙 분석

In [ ]:
#step 1. 라이브러리 호출
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from konlpy.tag import Okt
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 (Windows 기준 — Mac/Linux는 본인 환경에 맞는 폰트 경로로 수정 필요)
font_location = r'C:\Windows\Fonts\malgun.ttf'
font_name = fm.FontProperties(fname=font_location).get_name()
matplotlib.rc('font', family=font_name)

font_path = font_location


In [ ]:
# step 2. 파일 불러오기 (경로는 실행 환경에 맞게 수정)
kr = pd.read_csv('kr_sentiment.csv')
kr = kr[kr['review_clean'].notna()]
kr = kr[kr['review_clean'] != ''].reset_index(drop=True)
print(f'- 전체 리뷰 수 : {len(kr):,}건')
print(f'- 고유 제품 수 : {kr["product_name"].nunique()}개')


In [ ]:
# step 3. sentiment_score >= 0.8 필터링
kr_filtered = kr[kr['sentiment_score'] >= 0.8].copy()
print(f'- 필터링 후 리뷰 수 : {len(kr_filtered):,}건')
print(f"긍정 : {(kr_filtered['sentiment_label'] == '긍정').sum():,}건")
print(f"부정 : {(kr_filtered['sentiment_label'] == '부정').sum():,}건")


In [ ]:
# step 4. 불용어 + 정규화 사전 적용 (반복 탐색 끝에 확정된 최종 버전)
okt = Okt()

# 불용어 (별도 파일 필요 - 경로는 실행 환경에 맞게 수정)
with open('stopwords.txt', 'r', encoding='utf-8') as f:
    stopwords = set(f.read().splitlines())

# 정규화 사전
norm_groups = {
    '발림성'  : {'발라', '발랏', '발랏는', '발랏는데', '발랏다', '발랏을때',
                 '발랏을땬', '발랴', '발리', '발리나', '발린', '발림', '발링',
                 '뻑벅한', '뻑뻑', '뻑하거',
                 '밀리', '밀어내기', '밀림'},
    '눈시림'  : {'시려웠', '시렵거', '시렵던', '시리', '시림'},
    '밀착력'  : {'밀착', '밀착렫', '밀철'},
    '끈적임'  : {'끈', '끈끈', '끈끈함', '끈덕', '끈덕끈', '끈쩍', '끈쩍끈쩍', '끈쩍임', '적임'},
    '화끈거림': {'화끈', '화끈거렸', '후끈'},
    '백탁'    : {'탁', '백탁현상', '백탁'},
    '가성비'  : {'가성', '저렴', '가격'},
    '무기자차': {'무기'},
    '유기자차': {'유기'},
    '유지력'  : {'유지', '유지됨', '지속'},
    '흡수력'  : {'흡수'},
    '트러블'  : {'뽀로지', '뾰루지', '뽀루지'},
    '피부타입': {'지성만', '지성은', '지성이면', '지성인',
                 '부지', '건성', '복합', '지성'},
    '쿨링감'  : {'쿨롱', '쿨링', '쿨링금'},
    '보습력'  : {'촉촉', '수분', '보습'},
    '톤업' : {'톤업', '톤', '보정', '커버', '베이스', '프리'},
    '자외선차단' : {'차단', '자외선'},
    '향냄새' : {'향', '냄새'}
}

norm_dict = {}
for target, sources in norm_groups.items():
    for source in sources:
        norm_dict[source] = target
print(f'정규화 사전: {len(norm_dict)}개 항목')

# step 5. 토크나이저
def okt_tokenizer(text):
    morphs = okt.pos(str(text), stem=True, norm=True)
    tokens = []
    for word, pos in morphs:
        if pos in ['Noun']:
            word = norm_dict.get(word, word)
            if word not in stopwords:
                tokens.append(word)
    return tokens

# step 6. 트랜잭션 생성 함수
def get_transactions(df, sentiment=None, min_keywords=2):
    if sentiment:
        df = df[df['sentiment_label'] == sentiment]

    transactions = []
    for text in df['review_clean']:
        tokens = okt_tokenizer(text)
        tokens = list(set(tokens))
        if len(tokens) >= min_keywords:
            transactions.append(tokens)
    label = sentiment if sentiment else '전체'
    print(f'{label} 트랜잭션 : {len(transactions):,}건')
    return transactions

# step 7. Apriori 분석 함수
def run_apriori(transactions, min_support=0.03, min_confidence=0.3, min_lift=1.0):
    # min_support - N% 이상 등장할 경우 조합 추출
    # min_confidence - N% 이상일 경우 규칙으로 인정
    # min_lift - 우연보다 얼마나 더 자주 같이 나오는지, 1.0 이상이면 의미있는 연관성

    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df_te = pd.DataFrame(te_array, columns=te.columns_)
    print(f'키워드 수 : {len(te.columns_)}개 / 트랜잭션 : {len(df_te):,}건')

    frequent_itemsets = apriori(
        df_te,
        min_support=min_support,
        use_colnames=True,
        max_len=3
    )
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
    print(f'빈번 아이템셋 : {len(frequent_itemsets)}개')

    if len(frequent_itemsets) == 0:
        print('⚠️ 빈번 아이템셋 없음 → min_support 낮추기')
        return None, None

    rules = association_rules(
        frequent_itemsets,
        metric='confidence',
        min_threshold=min_confidence
    )
    rules = rules[rules['lift'] >= min_lift]
    rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
    print(f'연관규칙 : {len(rules)}개')
    return frequent_itemsets, rules

# 결과 출력 함수
def print_rules(rules, top_n=20):
    top_rules = rules.head(top_n).copy()
    top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ','.join(list(x)))
    top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ','.join(list(x)))

    result = top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
    result['support'] = result['support'].round(3)
    result['confidence'] = result['confidence'].round(3)
    result['lift'] = result['lift'].round(2)

    result = result.reset_index(drop=True)
    result.index += 1
    display(result)


### 국내 리뷰 (전체) 연관규칙 분석

In [ ]:
# ── 전체 긍정 리뷰 연관규칙 ───────────────────────────
print('전체 긍정 리뷰 연관규칙 분석')
print('='*50)
pos_transactions = get_transactions(kr_filtered, sentiment='긍정')
fi_pos, rules_pos = run_apriori(
    pos_transactions,
    min_support=0.04,
    min_confidence=0.25,
    min_lift=1.0
)
if rules_pos is not None:
    print_rules(rules_pos, top_n=20)

# ── 전체 부정 리뷰 연관규칙 ───────────────────────────
print('\n전체 부정 리뷰 연관규칙 분석')
print('='*50)
neg_transactions = get_transactions(kr_filtered, sentiment='부정')
fi_neg, rules_neg = run_apriori(
    neg_transactions,
    min_support=0.02,
    min_confidence=0.2,
    min_lift=1.0
)
if rules_neg is not None:
    print_rules(rules_neg, top_n=20)


## 2. 글로벌(GB) 연관규칙 분석

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

lemmatizer = WordNetLemmatizer()

# ── GB step 1. 파일 불러오기 (경로는 실행 환경에 맞게 수정) ──
gb = pd.read_csv('gb_sentiment.csv')
gb = gb[gb['review_clean'].notna()]
gb = gb[gb['review_clean'] != ''].reset_index(drop=True)

gb_filtered = gb[gb['sentiment_score'] >= 0.8].copy()
print(f'필터링 후: {len(gb_filtered):,}건')
print(f'긍정: {(gb_filtered["sentiment_label"]=="긍정").sum():,}건')
print(f'부정: {(gb_filtered["sentiment_label"]=="부정").sum():,}건')


In [ ]:
# ── GB step 2. 불용어 + 정규화 사전 (반복 탐색 끝에 확정된 최종 버전) ──
with open('stopwords_gb.txt', 'r', encoding='utf-8') as f:
    base_stopwords = set(f.read().splitlines())
    base_stopwords = {w.strip() for w in base_stopwords if w.strip()}

with open('stopwords_gb_sentiment.txt', 'r', encoding='utf-8') as f:
    sentiment_stopwords = set(f.read().splitlines())
    sentiment_stopwords = {w.strip() for w in sentiment_stopwords if w.strip()}

# 연관규칙용 = base + sentiment 모두 제거
arm_stopwords = base_stopwords | sentiment_stopwords
print(f'연관규칙용 불용어: {len(arm_stopwords)}개')

en_norm_groups = {
    'white_cast'   : {'white', 'cast', 'whitecast', 'casting'},
    'lightweight'  : {'light', 'lightweighted', 'litghweight', 'weightless'},
    'sticky'       : {'stickiness', 'tacky', 'stick', 'stickey', 'stickness'},
    'greasy'       : {'greasiness', 'grease'},
    'moisturizing' : {'moisturizes', 'moisturizer', 'moisturising', 'moisturiser', 'moisture', 'moist'},
    'hydrating'    : {'hydration', 'hydrated'},
    'scent'        : {'smell', 'fragrance'},
    'acne'         : {'breakout', 'pimple', 'blemish', 'prone', 'break', 'breakouts', 'outbreak'},
    'irritation'   : {'irritate', 'irritated', 'irritant', 'irritates', 'irritating', 'sting'},
    'packaging'    : {'package', 'pack'},
    'smooth'       : {'smoothly', 'spread', 'blend'},
    'oily'         : {'oil', 'oiler', 'oiliness', 'oily'},
    'transparency' : {'transparent'},
    'brightening'  : {'bright', 'brightens', 'brighter', 'brightness'},
    'absorption'   : {'absorb', 'absorbance', 'absorbed', 'absorbency', 'absorbent',
                       'absorber', 'absorbes', 'absorbing', 'absorbs'},
    'glowy'        : {'glow', 'glowy', 'shiny', 'shine', 'glowing'},
    'repurchase'   : {'repeat', 'repurchase'},
    'coverage'     : {'coverage', 'cover'},
    'finish'       : {'matte', 'dewy'},
    'price'        : {'price', 'affordable', 'cheap', 'expensive'},
    'protection'   : {'protects', 'protective'},
    'heavy'        : {'thick'},
    'watery'       : {'runny', 'liquid'},
    'pore'         : {'clog', 'cloggers', 'clogging'},
    'texture'      : {'formula', 'consistency'},
    'color'        : {'shade', 'tinted', 'pale', 'darker', 'yellow', 'purple'}
}

en_norm_dict = {}
for target, sources in en_norm_groups.items():
    for source in sources:
        en_norm_dict[source] = target

print(f'정규화 사전: {len(en_norm_dict)}개 항목')


In [ ]:
# ── step 3. 토크나이저 ────────────────────────────────
def en_tokenizer(text):
    tokens = word_tokenize(str(text).lower())
    tagged = pos_tag(tokens)
    result = []
    for word, tag in tagged:
        if tag.startswith('NN') or tag.startswith('JJ'):
            if tag.startswith('NN'):
                lemma = lemmatizer.lemmatize(word, pos='n')
            else:
                lemma = lemmatizer.lemmatize(word, pos='a')
            lemma = en_norm_dict.get(lemma, lemma)
            if len(lemma) >= 2 and lemma not in arm_stopwords:
                result.append(lemma)
    return result

# ── step 4. 트랜잭션 생성 함수 ───────────────────────
def get_transactions_gb(df, sentiment=None, min_keywords=2):
    if sentiment:
        df = df[df['sentiment_label'] == sentiment]

    transactions = []
    for text in df['review_clean']:
        tokens = en_tokenizer(text)
        tokens = list(set(tokens))
        if len(tokens) >= min_keywords:
            transactions.append(tokens)

    label = sentiment if sentiment else '전체'
    print(f'{label} 트랜잭션: {len(transactions):,}건')
    return transactions

# ── step 5. Apriori 분석 함수 ─────────────────────────
def run_apriori(transactions, min_support=0.05, min_confidence=0.3, min_lift=1.0):
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df_te = pd.DataFrame(te_array, columns=te.columns_)
    print(f'키워드 수: {len(te.columns_)}개 / 트랜잭션: {len(df_te):,}건')

    frequent_itemsets = apriori(df_te, min_support=min_support, use_colnames=True, max_len=3)
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
    print(f'빈번 아이템셋: {len(frequent_itemsets)}개')

    if len(frequent_itemsets) == 0:
        print('⚠️ 빈번 아이템셋 없음 → min_support 낮추기')
        return None, None

    rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=min_confidence)
    rules = rules[rules['lift'] >= min_lift]
    rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
    print(f'연관규칙: {len(rules)}개')
    return frequent_itemsets, rules

# ── step 6. 결과 출력 함수 ────────────────────────────
def print_rules(rules, top_n=20):
    top_rules = rules.head(top_n).copy()
    top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))

    result = top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
    result['support']    = result['support'].round(3)
    result['confidence'] = result['confidence'].round(3)
    result['lift']       = result['lift'].round(2)

    result = result.reset_index(drop=True)
    result.index += 1
    display(result)


In [ ]:
# ── GB 연관규칙 분석 실행 ─────────────────────────────
print('\n전체 긍정 리뷰 연관규칙 분석')
print('='*50)
pos_transactions_gb = get_transactions_gb(gb_filtered, sentiment='긍정')
fi_pos_gb, rules_pos_gb = run_apriori(
    pos_transactions_gb,
    min_support=0.04,
    min_confidence=0.25,
    min_lift=1.0
)
if rules_pos_gb is not None:
    print_rules(rules_pos_gb, top_n=20)

print('\n전체 부정 리뷰 연관규칙 분석')
print('='*50)
neg_transactions_gb = get_transactions_gb(gb_filtered, sentiment='부정')
fi_neg_gb, rules_neg_gb = run_apriori(
    neg_transactions_gb,
    min_support=0.03,   # 부정 리뷰 적으니까 낮게
    min_confidence=0.2,
    min_lift=1.0
)
if rules_neg_gb is not None:
    print_rules(rules_neg_gb, top_n=20)


## 3. 네트워크 시각화 (연관규칙 결과를 엣지·노드로 활용)

In [ ]:
import networkx as nx
from pyvis.network import Network
import math
import os


In [ ]:
# 연관규칙 네트워크 시각화 함수
def rules_to_pyvis(
    rules,
    out_html,
    title='연관규칙 네트워크',
    node_color='#5BA4E6',
    min_lift=1.0,
    min_conf=0.2,
    top_k=80,
    height='700px',
    merge_bidirectional=True,
    font_face='Arial',
):
    # ── 필터 + 정렬 ──────────────────────────────────
    rr = rules.copy()
    rr = rr[(rr['lift'] >= min_lift) & (rr['confidence'] >= min_conf)]
    rr = rr.sort_values(['lift', 'confidence', 'support'], ascending=False).head(top_k)

    if len(rr) == 0:
        print(f'⚠️ {title}: 조건에 맞는 규칙 없음 → 스킵')
        return None

    # ── 양방향 중복 제거 ──────────────────────────────
    if merge_bidirectional:
        r = rr.copy()
        r['a'] = r['antecedents'].apply(
            lambda s: list(s)[0] if len(s) == 1 else str(sorted(list(s)))
        )
        r['b'] = r['consequents'].apply(
            lambda s: list(s)[0] if len(s) == 1 else str(sorted(list(s)))
        )
        r['pair_key'] = r.apply(lambda x: tuple(sorted([x['a'], x['b']])), axis=1)
        r = r.sort_values('lift', ascending=False)
        r = r.drop_duplicates(subset=['pair_key'], keep='first')
        rr = r.drop(columns=['a', 'b', 'pair_key'])

    print(f'{title}: {len(rr)}개 규칙 사용')

    # ── 그래프 생성 ───────────────────────────────────
    G = nx.DiGraph()

    def as_label(x):
        return ' + '.join(sorted(list(x)))

    for _, row in rr.iterrows():
        a = as_label(row['antecedents'])
        c = as_label(row['consequents'])
        support = float(row['support'])
        conf = float(row['confidence'])
        lift = float(row['lift'])
        G.add_node(a)
        G.add_node(c)
        G.add_edge(a, c,
                   weight=math.log1p(lift),
                   title=f"지지도={support:.3f}<br>신뢰도={conf:.3f}<br>향상도={lift:.3f}")

    # ── PyVis 시각화 ──────────────────────────────────
    net = Network(height=height, width='100%',
                  directed=True, bgcolor='#ffffff', font_color='#333333')
    net.barnes_hut()

    deg = dict(G.degree())
    max_deg = max(deg.values()) if deg else 1

    for node in G.nodes():
        size = 15 + 30 * (deg.get(node, 0) / max_deg)
        net.add_node(
            node, label=node, size=size, color=node_color,
            title=f"{node}<br>연결 수={deg.get(node, 0)}"
        )

    edge_weights = nx.get_edge_attributes(G, 'weight')
    max_w = max(edge_weights.values()) if edge_weights else 1

    for u, v, data in G.edges(data=True):
        width = 1 + 6 * (data['weight'] / max_w)
        net.add_edge(u, v, title=data['title'], value=width,
                     color='rgba(120,120,120,0.5)')

    net.set_options(f"""
    var options = {{
        "nodes": {{
            "shape": "dot",
            "font": {{"size": 15, "face": "{font_face}"}}
        }},
        "edges": {{
            "arrows": {{"to": {{"enabled": true}}}},
            "smooth": {{"type": "dynamic"}}
        }},
        "interaction": {{"hover": true, "tooltipDelay": 100}},
        "physics": {{
            "enabled": true,
            "stabilization": {{"iterations": 300}},
            "barnesHut": {{
                "gravitationalConstant": -8000,
                "springLength": 180,
                "springConstant": 0.02,
                "damping": 0.6
            }}
        }}
    }}
    """)

    net.write_html(out_html, open_browser=False)
    print(f'✅ 저장: {out_html} (노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개)')
    return out_html


In [ ]:
# 공출현 네트워크 시각화 - 연관규칙 조건을 만족하는 데이터가 너무 적을 때(GB 부정 등) 대체용
def build_cooccurrence_network(transactions, title, out_html,
                                node_color='#5BA4E6',
                                top_n=30, min_edge=2):

    node_cnt = Counter()
    edge_cnt = Counter()

    for words in transactions:
        node_cnt.update(words)
        for i in range(len(words)):
            for j in range(i+1, len(words)):
                a, b = sorted([words[i], words[j]])
                edge_cnt[(a, b)] += 1

    top_nodes = {w for w, _ in node_cnt.most_common(top_n)}

    G = nx.Graph()
    for w in top_nodes:
        G.add_node(w, freq=node_cnt[w])
    for (a, b), cnt in edge_cnt.items():
        if cnt >= min_edge and a in top_nodes and b in top_nodes:
            G.add_edge(a, b, weight=cnt,
                       title=f"{a} — {b}<br>함께 등장: {cnt}회")

    if G.number_of_edges() == 0:
        print(f'⚠️ {title}: 엣지 없음')
        return

    net = Network(height='700px', width='100%',
                  directed=False, bgcolor='#ffffff',
                  font_color='#333333')

    # 물리엔진 끄기 → 안움직임
    net.toggle_physics(False)

    # spring layout으로 초기 배치
    pos = nx.spring_layout(G, seed=42, k=2.5)

    deg = dict(G.degree())
    max_deg = max(deg.values()) if deg else 1

    for node, data in G.nodes(data=True):
        freq = data.get('freq', 1)
        size = 8 + 12 * (deg.get(node, 0) / max_deg)
        x, y = pos[node]
        net.add_node(
            node,
            label=node,
            size=size,
            color=node_color,
            title=f"키워드: {node}<br>등장: {freq}회<br>연결: {deg.get(node,0)}개",
            x=float(x * 600),
            y=float(y * 600)
        )

    weights = nx.get_edge_attributes(G, 'weight')
    max_w = max(weights.values()) if weights else 1
    for u, v, data in G.edges(data=True):
        width = 0.5 + 3 * (data['weight'] / max_w)
        net.add_edge(u, v, title=data['title'], value=width,
                     color='rgba(100,100,100,0.6)')

    net.set_options("""
    var options = {
        "nodes": {
            "shape": "dot",
            "font": {"size": 12, "face": "Arial"}
        },
        "edges": {
            "smooth": {"type": "continuous"},
            "color": {"color": "rgba(150,150,150,0.35)"}
        },
        "interaction": {
            "hover": true,
            "dragNodes": true,
            "tooltipDelay": 100
        },
        "physics": {
            "enabled": true,
            "stabilization": {
                "enabled": true,
                "iterations": 200,
                "updateInterval": 50
            },
            "barnesHut": {
                "gravitationalConstant": -3000,
                "springLength": 120,
                "springConstant": 0.03,
                "damping": 0.5
            }
        }
    }
    """)

    net.write_html(out_html, open_browser=False)
    print(f'✅ {title} 저장 (노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개)')


In [ ]:
# ── 네트워크 시각화용 파라미터 재조정 + 실행 ─────────
# 표(연관규칙 결과)보다 완화된 기준을 사용 (그래프에 표시할 노드/엣지 확보 목적)
print('=== KR 긍정 ===')
fi_pos2, rules_pos2 = run_apriori(
    pos_transactions,
    min_support=0.03,
    min_confidence=0.15,
    min_lift=1.0
)
print_rules(rules_pos2, top_n=20)

print('\n=== KR 부정 ===')
fi_neg2, rules_neg2 = run_apriori(
    neg_transactions,
    min_support=0.02,
    min_confidence=0.15,
    min_lift=1.0
)
print_rules(rules_neg2, top_n=20)

print('\n=== GB 긍정 ===')
fi_pos_gb2, rules_pos_gb2 = run_apriori(
    pos_transactions_gb,
    min_support=0.03,
    min_confidence=0.15,
    min_lift=1.0
)
print_rules(rules_pos_gb2, top_n=20)

# ── 네트워크 저장 경로 (경로는 실행 환경에 맞게 수정) ──
base = './network_analysis'
os.makedirs(base, exist_ok=True)

# ── 네트워크 생성 ─────────────────────────────────────
# KR 긍정
rules_to_pyvis(
    rules_pos2,
    f'{base}/kr_pos_network.html',
    title='KR 긍정 리뷰 연관 네트워크',
    node_color='#5BA4E6',
    min_lift=1.0, min_conf=0.15
)

# KR 부정
rules_to_pyvis(
    rules_neg2,
    f'{base}/kr_neg_network.html',
    title='KR 부정 리뷰 연관 네트워크',
    node_color='#E87070',
    min_lift=1.0, min_conf=0.15
)

# GB 긍정
rules_to_pyvis(
    rules_pos_gb2,
    f'{base}/gb_pos_network.html',
    title='GB 긍정 리뷰 연관 네트워크',
    node_color='#5BA4E6',
    min_lift=1.0, min_conf=0.15
)

# GB 부정 → 공출현 네트워크 (리뷰 수가 너무 적어 연관규칙 시각화에 한계)
build_cooccurrence_network(
    neg_transactions_gb, 'GB 부정',
    f'{base}/gb_neg_cooc.html',
    node_color='#E87070',
    top_n=12, min_edge=3
)

print('\n=== 전체 완료 ===')
print(f'KR 긍정 → kr_pos_network.html  (규칙 {len(rules_pos2)}개)')
print(f'KR 부정 → kr_neg_network.html  (규칙 {len(rules_neg2)}개)')
print(f'GB 긍정 → gb_pos_network.html  (규칙 {len(rules_pos_gb2)}개)')
print(f'GB 부정 → gb_neg_cooc.html     (공출현, 데이터 한계)')
